In [5]:
from datasets import load_dataset
import random
from collections import defaultdict
import regex as re
from __future__ import annotations

ds = load_dataset("roneneldan/TinyStories")['train']
text = "<|endoftext|>".join(ds[0:10000]['text'])
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
special_tokens = ["<|endoftext|>"]
vocab_limit = 1000



In [6]:
# return counts of pre_tokens as a dictionary
def pre_tokenization(text: str, PAT: str, special_tokens: list[str]) -> dict[str]:
    i = 0
    pre_tokens_str = defaultdict(int)
    st_PAT = "|".join([re.escape(x) for x in special_tokens])
    for st_match in re.finditer(st_PAT, text):
        j = st_match.start()
        ss = text[i:j]
        for match in re.finditer(PAT, ss):
            word = match.group()
            pre_tokens_str[word] += 1
        i = st_match.end()
    ss = text[i:]
    for match in re.finditer(PAT, ss):
        word = match.group()
        pre_tokens_str[word] += 1
    return pre_tokens_str



pre_tokens = pre_tokenization(text, PAT, special_tokens)
print(len(pre_tokens))

12085


In [7]:
# update bp count and pre_tokens as a result of bp merge
def update_bp_count(bp, bp_count, pre_tokens):
    for ini_word in list(pre_tokens.keys()):
        i = 0
        count = pre_tokens[ini_word]
        word = ini_word
        # scan through the word to look for the bp. if found, update the word and bp_count influenced by the merge
        while i < len(word)-1:
            if word[i]==bp[0] and word[i+1]==bp[1]:
                if i > 0:
                    bp_count[(word[i-1], bp[0]+bp[1])] += count
                    bp_count[(word[i-1], bp[0])] -= count
                if i+2 < len(word):
                    bp_count[(bp[0]+bp[1], word[i+2])] += count
                    bp_count[(bp[1], word[i+2])] -= count
                bp_count[(bp[0], bp[1])] -= count
                word = word[:i] + (bp[0]+bp[1],) + word[i+2:]
            i += 1

        # update tokenization of the pretoken / word
        if word != ini_word:
            pre_tokens.pop(ini_word)
            pre_tokens[word] = count
            


def construct_bpe(pre_tokens_str: dict[str],  vocab_size: int, special_tokens: list[str]):
    pre_tokens = dict()
    for word, count in pre_tokens_str.items():
        word_tuple_of_bytes = tuple(bytes([a]) for a in word.encode("utf-8"))
        pre_tokens[word_tuple_of_bytes] = count
    
    # construct initial vocab
    token_id = 0
    vocab = dict()
    for i in range(256):
        vocab[token_id] = bytes([i])
        token_id += 1
    for t in special_tokens:
        vocab[token_id] = t.encode("utf-8")
        token_id += 1
    

    # construct initial bytes-pair count
    bp_count = defaultdict(int)
    for token, count in pre_tokens.items():
        for i in range(len(token)-1):
            bp = (token[i], token[i+1])
            bp_count[bp] += count

    # merge
    merges = []
    while len(vocab) < vocab_size:
        # find the most frequent bp, if there is a tie then pick the first based on lexi order
        max_count = max(bp_count.values())
        tmp = [bp for bp, count in bp_count.items() if count==max_count]
        bp = max(tmp)
        merges.append(bp)

        # add it to vocab
        vocab[token_id] = bp[0] + bp[1]
        token_id += 1

        # update bp count and tokenization of pre-tokens
        update_bp_count(bp, bp_count, pre_tokens)
        assert(bp_count[bp]==0)
        
    return vocab, merges


vocab, merges = construct_bpe(pre_tokens, vocab_limit, special_tokens)

In [8]:
for i in range(256, 256 + 20):
    try:
        print(vocab[i].decode("utf-8"))
    except:
        continue

<|endoftext|>
he
 t
 a
 s
 w
nd
 the
ed
 and
in
 to
 b
 h
 wa
re
ou
 f
it
 l


In [12]:
class Tokenizer():
    def __init__(self, vocab, merges, special_tokens, PAT=None):
        # bidirectional mapping between ids and tokens
        self.id2token = vocab
        self.token2id = defaultdict(None)
        for i, token in self.id2token.items():
            self.token2id[token] = i
        
        # ordered set of merges for fast query
        self.merges = dict()
        for i in range(len(merges)):
            self.merges[merges[i]] = i
        
        self.st_PAT = "|".join([re.escape(x) for x in special_tokens])
        if PAT:
            self.PAT = PAT
        else:
            self.PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    
    def _encode_pretoken(self, bs: bytes):
        if bs in self.token2id:
            return [self.token2id[bs]]
        
        word = tuple(bytes([b]) for b in bs)

        while True:
            matches = []
            for i in range(len(word)-1):
                bp = word[i] + word[i+1]
                if bp in self.merges:
                    matches.append((self.merges[bp], bp, i))
            if not matches:
                break
            bp, i = min(matches)[1:2]
            word = word[:i] + (bp,) + word[i+2:]
        
        return [self.token2id[t] for t in word]
    
    
    # turn a string without special token into pre-token list
    def encode(self, s):
        result = []

        i = 0
        for st_match in re.finditer(self.st_PAT, s):
            j = st_match.start()
            ss = s[i:j]
            for match in re.finditer(self.PAT, ss):
                bs = match.group().encode("utf-8")
                result.extend(self._encode_pretoken(bs))
            st_bs = st_match.group().encode("utf-8")
            result.append(self.token2id[st_bs])
            i = st_match.end()
        ss = s[i:]
        for match in re.finditer(self.PAT, ss):
            bs = match.group().encode("utf-8")
            result.extend(self._encode_pretoken(bs))
        return result
    
    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        for s in iterable:
            id_list = self.encode(s)
            for i in id_list:
                yield i

    
    def decode(self, token_ids):
        text = b"".join(self.id2token[i] for i in token_ids).decode("utf-8")
        return text


special_tokens = ["<|endoftext|>"]
tokenizer = Tokenizer(vocab, merges, special_tokens)

text = "<|endoftext|>".join([get_random_string(100) for _ in range(10)])
token_ids = tokenizer.encode(text)
ttext = tokenizer.decode(token_ids)
assert(ttext == text)

token_ids = [random.randint(0, vocab_limit - 1) for _ in range(1000)]
ttext = tokenizer.decode(token_ids)
ttoken_ids = tokenizer.encode(ttext)
assert(token_ids == ttoken_ids)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xbf in position 16: invalid start byte

In [10]:
import random

def get_random_string(length):
    # Unicode 最大码位是 0x10FFFF
    # 0xD800 - 0xDFFF 是代理对区域，必须避开，否则会报错
    valid_ranges = [
        (0x0020, 0xD7FF),  
        (0xE000, 0x10FFFF) 
    ]
    
    res = []
    for _ in range(length):
        r = random.choice(valid_ranges)
        code_point = random.randint(*r)
        res.append(chr(code_point))
    
    return "".join(res)

In [29]:
text

'\u0557\ua4c9\U00087f33\U00071160\U000fc993菸휠볗查퓤줳轉໘\U0008a83c\U0004852a䧀渿均ᆬ\U0008c37fÅ\U00108e4a\U00038ddd\U001075fb鋊褯\U0003a641\U00104693鲱㖯\U000e6a30\U000b8162ຂ誢\U0001fc0fꋊ挿蒚\U000f771cޯⴧ\U000d9861𭐀먦쳯\U00098377\U000d8ed0嚮𩠢\U000323f6Ϭ\U00050477𓅤\U000608fc\U0003771d\U0006d125塢၈\U0006d894ꪶ\U0003cf96৲\U0004c006\U0010145c\U0008d4c3\U00015f9b췜\U0004ea89\u1ad2敘\U0010977d\U0006ebcbꂝ\U000b4937Ӫ걎\U00048d2f\U0006b391\U000d1dc7\U00100d91\U000c7eb9祩\U000952b8\U0004b94e\U00063147\U000333a0왐遧\U00043a4bဏ𗜿骞⥫\U000d8d4bხ\U000f0fea♁ꎬ䗋\U0003c32c'